In [1]:
!apt-get update
!apt-get install -y libopencv-dev build-essential cmake

Hit:1 http://archive.ubuntu.com/ubuntu noble InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 http://archive.ubuntu.com/ubuntu noble-updates InRelease [126 kB]
Get:4 http://security.ubuntu.com/ubuntu noble-security InRelease [126 kB]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ InRelease [3,631 B]
Get:6 http://archive.ubuntu.com/ubuntu noble-backports InRelease [126 kB]
Get:7 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu noble InRelease [17.8 kB]
Get:8 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Get:9 https://r2u.stat.illinois.edu/ubuntu noble InRelease [9,161 B]
Get:10 https://cloud.r-project.org/bin/linux/ubuntu noble-cran40/ Packages [78.0 kB]
Get:11 http://security.ubuntu.com/ubuntu noble-security/main amd64 Packages [1,412 kB]
Get:12 http://security.ubuntu.com/ubuntu noble-security/universe amd64 Packages [1,550 kB]
Get:13 http://archive.ubuntu.com/ubuntu noble-updates/universe amd64 Packages [2,17

In [2]:
%%writefile main.cpp
#include <iostream>
#include <opencv2/opencv.hpp>
#include <cuda_runtime.h>

int main() {
    std::cout << "OpenCV Versiyonu: " << CV_VERSION << std::endl;

    // Colab GPU Bağlantı Kontrolü
    int deviceCount = 0;
    cudaError_t error = cudaGetDeviceCount(&deviceCount);

    if (error == cudaSuccess && deviceCount > 0) {
        std::cout << "Başarılı! Colab GPU (CUDA) Algılandı. Cihaz Sayısı: " << deviceCount << std::endl;
    } else {
        std::cout << "CUDA Cihazı Bulunamadı veya GPU Aktif Edilmedi!" << std::endl;
    }

    return 0;
}

Writing main.cpp


In [3]:
!g++ main.cpp -o app `pkg-config --cflags --libs opencv4` -I/usr/local/cuda/include -L/usr/local/cuda/lib64 -lcudart
!./app

main.cpp:3:10: fatal error: cuda_runtime.h: No such file or directory
    3 | #include <cuda_runtime.h>
      |          ^~~~~~~~~~~~~~~~
compilation terminated.
/bin/bash: line 1: ./app: No such file or directory


In [4]:
%%writefile main.cpp
#include <iostream>
#include <vector>
#include <complex>
#include <cmath>
#include <iomanip>
#include <opencv2/opencv.hpp>

// 2'nin kuvveti uzunluktaki bir vektör için DFT fonksiyonu
std::vector<std::complex<double>> hesaplaDFT(const std::vector<double>& x) {
    int N = x.size();
    std::vector<std::complex<double>> X(N, 0);
    const double PI = std::acos(-1.0);

    for (int k = 0; k < N; ++k) {
        std::complex<double> sum(0.0, 0.0);
        for (int n = 0; n < N; ++n) {
            double angle = 2 * PI * k * n / N;
            std::complex<double> w(std::cos(angle), -std::sin(angle));
            sum += x[n] * w;
        }
        X[k] = sum;
    }
    return X;
}

int main() {
    std::cout << "=== OpenCV Versiyon Kontrolu ===" << std::endl;
    std::cout << "OpenCV Versiyonu: " << CV_VERSION << std::endl;

    // Görsel yükleme testi
    cv::Mat img = cv::imread("hafta7-1.jpeg", cv::IMREAD_GRAYSCALE);
    if(img.empty()) {
        std::cout << "Gorsel yuklenemedi! Lutfen /content/hafta7-1.jpeg yolunu kontrol edin." << std::endl;
    } else {
        std::cout << "Gorsel Basariyla Yuklendi. Boyutlar: " << img.cols << "x" << img.rows << std::endl;
    }

    std::cout << "\n=== DFT ve Harmonik Analiz (N = 8, 2'nin Kuvveti) ===" << std::endl;

    // Örnek sinyal: Temiz (Gürültüsüz) Sinyal
    // x[n] = sin(2 * pi * n / 8) -> 1. harmonikte tepe yapması beklenir
    int N = 8;
    std::vector<double> temiz_sinyal(N);
    std::vector<double> gurultulu_sinyal(N);
    const double PI = std::acos(-1.0);

    for (int n = 0; n < N; ++n) {
        temiz_sinyal[n] = std::sin(2 * PI * n / N);
        // Rastgele ufak bir gürültü ekleyelim
        double gurultu = ((double)rand() / RAND_MAX - 0.5) * 0.4;
        gurultulu_sinyal[n] = temiz_sinyal[n] + gurultu;
    }

    // DFT hesaplamaları
    auto dft_temiz = hesaplaDFT(temiz_sinyal);
    auto dft_gurultulu = hesaplaDFT(gurultulu_sinyal);

    // Sonuçların yazdırılması ve Karşılaştırılması
    std::cout << std::fixed << std::setprecision(4);
    std::cout << std::left
              << std::setw(6) << "n/k"
              << std::setw(15) << "Temiz Sinyal"
              << std::setw(25) << "Temiz Sinyal DFT (Genlik)"
              << std::setw(18) << "Gurultulu Sinyal"
              << std::setw(25) << "Gurultulu Sinyal DFT (Genlik)"
              << std::endl;

    for (int i = 0; i < N; ++i) {
        double genlik_temiz = std::abs(dft_temiz[i]);
        double genlik_gurultulu = std::abs(dft_gurultulu[i]);
        std::cout << std::left
                  << std::setw(6) << i
                  << std::setw(15) << temiz_sinyal[i]
                  << std::setw(25) << genlik_temiz
                  << std::setw(18) << gurultulu_sinyal[i]
                  << std::setw(25) << genlik_gurultulu
                  << std::endl;
    }

    std::cout << "\nYorum: Gurultusuz sinyalde sadece ana frekans bilesenlerinde (k=1 ve simetrisi k=7) "
              << "net harmonik genlikleri gorulurken, gurultulu sinyalde gurultunun etkisiyle diger frekanslarda "
              << "da sıfırdan farklı genlik bilesenleri olusmustur." << std::endl;

    return 0;
}

Overwriting main.cpp


In [5]:
# CUDA kütüphanelerini bulamama sorununu çözmek için doğru path tanımlanarak derleme yapıyoruz
!g++ main.cpp -o app `pkg-config --cflags --libs opencv4` -I/usr/local/cuda/include -L/usr/local/cuda/lib64
!./app

=== OpenCV Versiyon Kontrolu ===
OpenCV Versiyonu: 4.6.0
Gorsel Basariyla Yuklendi. Boyutlar: 1012x472

=== DFT ve Harmonik Analiz (N = 8, 2'nin Kuvveti) ===
n/k   Temiz Sinyal   Temiz Sinyal DFT (Genlik)Gurultulu Sinyal  Gurultulu Sinyal DFT (Genlik)
0     0.0000         0.0000                   0.1361            0.4115                   
1     0.7071         4.0000                   0.6649            4.2434                   
2     1.0000         0.0000                   1.1132            0.4650                   
3     0.7071         0.0000                   0.8265            0.1376                   
4     0.0000         0.0000                   0.1647            0.2846                   
5     -0.7071        0.0000                   -0.8281           0.1376                   
6     -1.0000        0.0000                   -1.0659           0.4650                   
7     -0.7071        4.0000                   -0.5998           4.2434                   

Yorum: Gurultusuz sinyalde 